In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_directory = 'rca100/cases/t008'
task_file_path = f'{task_directory}/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')

alert_title, alert_window, alert_entity

```
Out[1]:
```
The alert is titled "cart响应时间抖动告警" (Cart Response Time Jitter Alert). It occurred between 04:22:01 and 04:28:03 on April 13, 2026 (UTC+8). The entity that triggered the alert is the operation `cart::POST /oteldemo.CartService/AddItem` of type `apm.operation` in the `apm` domain.

The original code execution output of IPython Kernel is also provided below for reference:

('cart响应时间抖动告警', {'start': '2026-04-13T04:22:01+08:00', 'end': '2026-04-13T04:28:03+08:00'}, {'entity_id': 'e433b3f842dc4e525a834701cc603c28', 'entity_name': 'cart::POST /oteldemo.CartService/AddItem', 'entity_type': 'apm.operation', 'entity_domain': 'apm'})```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = f'{task_directory}/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# List all available KPIs
available_kpis = metrics_data['metric'].unique()
available_kpis

```
Out[2]:
```
The available KPIs in the `metrics.parquet` file include a wide range of metrics related to nodes, deployments, clusters, namespaces, application performance, JVM, and thread pools. Key KPIs for anomaly detection and fault identification include `error_count`, `error_rate`, `slow_count`, `latency`, `request_count`, `node_cpu_usage_rate`, `node_memory_usage_rate`, `deployment_cpu_usage_total`, `deployment_memory_usage_total`, and `avg_request_latency_seconds`.

The original code execution output of IPython Kernel is also provided below for reference:

['node_ready_status' 'node_not_ready_status' 'node_memory_usage_rate'
 'node_disk_usage_rate' 'node_pod_running_count' 'node_cpu_usage_rate'
 'deployment_ready_replicas' 'deployment_available_replicas'
 'deployment_desired_replicas' 'deployment_availability_rate'
 'deployment_memory_usage_total' 'deployment_cpu_usage_total'
 'deployment_cpu_usage_vs_limits' 'deployment_cpu_usage_vs_requests'
 'cluster_failed_node_count' 'cluster_node_count'
 'cluster_number_of_not_ready_pods' 'apiserver_concurrent_connections'
 'apiserver_non_2xx_write_request_qps'
 'apiserver_non_2xx_read_request_qps' 'cluster_number_of_pod'
 'apiserver_requests_per_second' 'namespace_number_of_not_ready_pods'
 'namespace_number_of_pods' 'error_count' 'slow_count' 'request_count'
 'avg_request_latency_seconds' 'arms_jvm_gc_total'
 'arms_jvm_gc_seconds_total' 'arms_jvm_gc_delta'
 'arms_jvm_gc_seconds_delta' 'arms_jvm_mem_used_bytes'
 'arms_jvm_mem_init_bytes' 'arms_jvm_mem_committed_bytes'
 'arms_jvm_mem_max_bytes' 'arms_jvm_threads_count'
 'arms_thread_pool_active_thread_count'
 'arms_thread_pool_current_thread_count'
 'arms_thread_pool_max_thread_count' 'arms_thread_pool_max_pool_size'
 'arms_thread_pool_core_pool_size' 'arms_thread_pool_scheduled_task_count'
 'arms_thread_pool_completed_task_count'
 'arms_thread_pool_rejected_task_count' 'arms_thread_pool_queue_size'
 'arms_grouped_thread_count' 'arms_grouped_thread_cpu_time'
 'arms_grouped_thread_runnable_count' 'arms_grouped_thread_waiting_count'
 'arms_grouped_thread_timed_waiting_count' 'cpu' 'latency' 'mem'
 'workload' 'slow' 'error' 'error_rate']```
```

In [ ]:
# In[3]:

# Ensure 'time' and 'value' columns are numeric for calculations
metrics_data['time'] = pd.to_numeric(metrics_data['time'], errors='coerce')
metrics_data['value'] = pd.to_numeric(metrics_data['value'], errors='coerce')

# Group by component (entity_name) and KPI (metric), then calculate the global P95 threshold
p95_thresholds = metrics_data.groupby(['entity_name', 'metric'])['value'].quantile(0.95).reset_index()
p95_thresholds.columns = ['component', 'kpi', 'p95_threshold']

p95_thresholds

```
Out[3]:
```
The metrics data has been aggregated by each KPI for each component (e.g., services like `cart` and operations like `cart::POST /oteldemo.CartService/AddItem`). The global P95 threshold has been calculated for each 'component-KPI' time series across the entire dataset. This provides a benchmark to identify anomalies by comparing actual values against these thresholds. There are 272 unique 'component-KPI' combinations in total.

The original code execution output of IPython Kernel is also provided below for reference:

component                                  kpi  p95_threshold
0                                                     apiserver_concurrent_connections       6.000000
1                                                   apiserver_non_2xx_read_request_qps       0.267261
2                                                  apiserver_non_2xx_write_request_qps       0.053723
3                                                        apiserver_requests_per_second      21.246175
4                                                            cluster_failed_node_count       0.000000
..                                            ...                                  ...            ...
267                                      shipping                             workload    4668.600000
268   shipping::oteldemo.ShippingService/GetQuote          avg_request_latency_seconds       0.002863
269   shipping::oteldemo.ShippingService/GetQuote                        request_count    2334.000000
270  shipping::oteldemo.ShippingService/ShipOrder          avg_request_latency_seconds       0.000018
271  shipping::oteldemo.ShippingService/ShipOrder                        request_count    2334.600000

[272 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

from datetime import datetime
import pytz

# Define the alert time window in UTC+8
alert_start = datetime.fromisoformat('2026-04-13T04:22:01+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat('2026-04-13T04:28:03+08:00').astimezone(pytz.timezone('Asia/Shanghai'))

# Convert the alert time window to Unix microseconds for filtering
alert_start_us = int(alert_start.timestamp() * 1_000_000)
alert_end_us = int(alert_end.timestamp() * 1_000_000)

# Filter metrics data to include only the time window of the alert
filtered_metrics = metrics_data[(metrics_data['time'] >= alert_start_us) & (metrics_data['time'] <= alert_end_us)]

# Merge the filtered metrics with the P95 thresholds
anomalies = filtered_metrics.merge(p95_thresholds, left_on=['entity_name', 'metric'], right_on=['component', 'kpi'])

# Mark data points as anomalous if they exceed the P95 threshold
anomalies['is_anomalous'] = anomalies['value'] > anomalies['p95_threshold']

# Filter only the anomalous data points
anomalous_data = anomalies[anomalies['is_anomalous']]

anomalous_data

```
Out[4]:
```
During the alert time window (04:22:01 to 04:28:03 on April 13, 2026, UTC+8), 295 anomalous data points were identified across various components and KPIs. These anomalies were detected by comparing the metric values against their respective global P95 thresholds. Key components with anomalies include `node_disk_usage_rate` for Kubernetes nodes, and `latency`, `cpu`, and `mem` for services like `email`, `frontend-proxy`, `frontend`, `image-provider`, and `inventory`.

The original code execution output of IPython Kernel is also provided below for reference:

time domain          entity_set entity_id     entity_name  ... service       component                   kpi p95_threshold is_anomalous
650   1776025321000000    k8s            k8s.node                            ...    None                  node_disk_usage_rate  3.625248e+01         True
651   1776025351000000    k8s            k8s.node                            ...    None                  node_disk_usage_rate  3.625248e+01         True
652   1776025381000000    k8s            k8s.node                            ...    None                  node_disk_usage_rate  3.625248e+01         True
653   1776025411000000    k8s            k8s.node                            ...    None                  node_disk_usage_rate  3.625248e+01         True
654   1776025441000000    k8s            k8s.node                            ...    None                  node_disk_usage_rate  3.625248e+01         True
...                ...    ...                 ...       ...             ...  ...     ...             ...                   ...           ...          ...
8203  1776025681000000    apm  apm.service.legacy                     email  ...    None           email                   mem  5.914867e+08         True
8211  1776025681000000    apm  apm.service.legacy            frontend-proxy  ...    None  frontend-proxy               latency  3.188761e-02         True
8216  1776025681000000    apm  apm.service.legacy                  frontend  ...    None        frontend               latency  3.095042e-02         True
8220  1776025681000000    apm  apm.service.legacy            image-provider  ...    None  image-provider                   cpu  1.113966e-04         True
8224  1776025681000000    apm  apm.service.legacy                 inventory  ...    None       inventory               latency  1.342895e-03         True

[295 rows x 13 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Identify consecutive anomalies for each component-KPI combination
anomalous_data['time'] = pd.to_numeric(anomalous_data['time'], errors='coerce')  # Ensure 'time' is numeric
anomalous_data = anomalous_data.sort_values(by=['component', 'kpi', 'time'])  # Sort by component, KPI, and time

# Calculate time difference between consecutive anomalies for each component-KPI
anomalous_data['time_diff'] = anomalous_data.groupby(['component', 'kpi'])['time'].diff().fillna(0)

# Define a threshold for consecutive anomalies (e.g., 1 minute in microseconds)
consecutive_threshold = 60 * 1_000_000  # 1 minute in microseconds

# Mark consecutive anomalies
anomalous_data['is_consecutive'] = anomalous_data['time_diff'] <= consecutive_threshold

# Group consecutive anomalies into fault periods
anomalous_data['fault_group'] = (anomalous_data['is_consecutive'] == False).cumsum()

# Filter out isolated noise spikes (fault groups with only one anomaly)
fault_groups = anomalous_data.groupby(['component', 'kpi', 'fault_group']).size().reset_index(name='count')
significant_fault_groups = fault_groups[fault_groups['count'] > 1]

# Merge back to keep only significant faults
significant_faults = anomalous_data.merge(significant_fault_groups, on=['component', 'kpi', 'fault_group'])

significant_faults

```
Out[5]:
```
During the alert time window, significant faults were identified by grouping consecutive anomalies for each component-KPI combination. A total of 265 significant fault data points were detected, with isolated noise spikes filtered out. These faults represent sustained anomalies across various components and KPIs, indicating potential root causes of the issue.

The original code execution output of IPython Kernel is also provided below for reference:

time domain      entity_set                         entity_id                 entity_name  ... is_anomalous   time_diff is_consecutive fault_group count
0    1776025531000000    k8s     k8s.cluster                                                                ...         True         0.0           True           0     2
1    1776025591000000    k8s     k8s.cluster                                                                ...         True  60000000.0           True           0     2
2    1776025531000000    k8s     k8s.cluster                                                                ...         True         0.0           True           0     2
3    1776025591000000    k8s     k8s.cluster                                                                ...         True  60000000.0           True           0     2
4    1776025321000000    k8s  k8s.deployment                                                                ...         True         0.0           True           0    36
..                ...    ...             ...                               ...                         ...  ...          ...         ...            ...         ...   ...
260  1776025501000000    apm    apm.instance  23afb9d9bc31156788a5190e42bc1f7f  inventory-78697ff45b-7v69l  ...         True         0.0           True          14     5
261  1776025531000000    apm    apm.instance  23afb9d9bc31156788a5190e42bc1f7f  inventory-78697ff45b-7v69l  ...         True  30000000.0           True          14     5
262  1776025591000000    apm    apm.instance  23afb9d9bc31156788a5190e42bc1f7f  inventory-78697ff45b-7v69l  ...         True  60000000.0           True          14     5
263  1776025621000000    apm    apm.instance  23afb9d9bc31156788a5190e42bc1f7f  inventory-78697ff45b-7v69l  ...         True  30000000.0           True          14     5
264  1776025651000000    apm    apm.instance  23afb9d9bc31156788a5190e42bc1f7f  inventory-78697ff45b-7v69l  ...         True  30000000.0           True          14     5

[265 rows x 17 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Load the traces.parquet file
traces_file_path = f'{task_directory}/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Ensure 'startTime', 'endTime', and 'duration' are numeric
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')
traces_data['duration'] = pd.to_numeric(traces_data['duration'], errors='coerce')

# Convert alert time window to nanoseconds for filtering
alert_start_ns = alert_start_us * 1_000  # Convert microseconds to nanoseconds
alert_end_ns = alert_end_us * 1_000  # Convert microseconds to nanoseconds

# Filter traces within the alert time window
filtered_traces = traces_data[(traces_data['startTime'] >= alert_start_ns) & (traces_data['endTime'] <= alert_end_ns)]

# Filter traces involving the 'cart' service and its operation
cart_traces = filtered_traces[
    (filtered_traces['serviceName'] == 'cart') &
    (filtered_traces['spanName'] == 'POST /oteldemo.CartService/AddItem')
]

# Check for error traces (statusCode != 0 or statusMessage not null)
cart_error_traces = cart_traces[(cart_traces['statusCode'] != 0) | (cart_traces['statusMessage'].notnull())]

# Trace the call chain to identify the most downstream faulty service
downstream_faults = filtered_traces[filtered_traces['traceId'].isin(cart_error_traces['traceId'])]
downstream_faults = downstream_faults[downstream_faults['statusCode'] != 0]

cart_error_traces, downstream_faults

```
Out[6]:
```
During the alert time window, traces involving the `cart` service and its operation `POST /oteldemo.CartService/AddItem` were identified. Among these, error traces were detected, indicating issues in the operation. By tracing the call chain, the most downstream faulty service was identified, which includes operations like `SELECT inventory.inventory` and `/api/v1/inventory/{productId}` in the `inventory` service. These downstream faults suggest that the root cause may be related to the `inventory` service.

The original code execution output of IPython Kernel is also provided below for reference:

(                                 traceId            spanId      parentSpanId kind                            spanName  ...                                         attributes  __topic__      __tag__:__pack_id__ events links
299964  cffd134b03710378354d12b3505b4238  a06caaf96b7c72f9  e2426a288d2dfc27    2  POST /oteldemo.CartService/AddItem  ...  {"user_agent.original":"grpc-node-js/1.12.6","...             C8F27BF240B79DF1-583784   None  None
299965  91e414d5c77a1c0d7985ce50b2ee4758  150c8331b6d7e71f  f502f951cc2f208f    2  POST /oteldemo.CartService/AddItem  ...  {"user_agent.original":"grpc-node-js/1.12.6","...             C8F27BF240B79DF1-583784   None  None
299970  4e145455195946abe27324e2904e8df5  bc2e7f0ec6b5fd22  6a78a35deed46581    2  POST /oteldemo.CartService/AddItem  ...  {"user_agent.original":"grpc-node-js/1.12.6","...             C8F27BF240B79DF1-583784   None  None
299972  bfb8dd674d402857842236fff050ad87  8dc927f84593f424  973f891f9ed26f30    2  POST /oteldemo.CartService/AddItem  ...  {"user_agent.original":"grpc-node-js/1.12.6","...             C8F27BF240B79DF1-583784   None  None
299975  661ebc77e415bffdc46c86a510e326de  25fe00cc5f553829  7b0b8e7147c931e2    2  POST /oteldemo.CartService/AddItem  ...  {"user_agent.original":"grpc-node-js/1.12.6","...             C8F27BF240B79DF1-583784   None  None
...                                  ...               ...               ...  ...                                 ...  ...                                                ...        ...                      ...    ...   ...
476565  4e031d66027b56fe57b8f0490ac8ef54  d3f5ddd02557bf18  08638a3ba9f1e163    2  POST /oteldemo.CartService/AddItem  ...  {"user_agent.original":"grpc-node-js/1.12.6","...             C1BE3A542C9B64D3-58072B   None  None
476569  2947b770b202feb361845c5d55e778c1  22c6c285ca5bef06  9adad6d7664ea980    2  POST /oteldemo.CartService/AddItem  ...  {"user_agent.original":"grpc-node-js/1.12.6","...             C1BE3A542C9B64D3-58072B   None  None
476570  80bc5c2d2508095aeb570012abcd5947  1abbf84cc484de27  cadc0b130c1e8b0d    2  POST /oteldemo.CartService/AddItem  ...  {"user_agent.original":"grpc-node-js/1.12.6","...             C1BE3A542C9B64D3-58072B   None  None
476574  66ee0a6a9e29afc129c821ceeb382ee3  eb26591b61551b7c  157cd12d037141e8    2  POST /oteldemo.CartService/AddItem  ...  {"user_agent.original":"grpc-node-js/1.12.6","...             C1BE3A542C9B64D3-58072B   None  None
499504  f30cf0cd95a6190f2efd6c02e332fb8d  7dae3234d8660b86  eecc0b2e423c2170    2  POST /oteldemo.CartService/AddItem  ...  {"user_agent.original":"grpc-node-js/1.12.6","...             BAEAC2AE47649F72-58302A   None  None

[2301 rows x 21 columns],                                  traceId            spanId      parentSpanId kind                               spanName  ...                                         attributes  __topic__      __tag__:__pack_id__ events links
299500  cffd134b03710378354d12b3505b4238  2d8cdf8e6b3c7aae  5d0ec3ccd7a6cd54    3             SELECT inventory.inventory  ...  {"serviceType":"2100","otel.scope.name":"io.op...             F21C6CFACBEB01A3-581BB3   None  None
299501  af0abbbfe79f3e2d34376c1dd1b5e6ed  d9a4ece9035f48e7  ac17e00349bd7f0b    3             SELECT inventory.inventory  ...  {"serviceType":"2100","otel.scope.name":"io.op...             F21C6CFACBEB01A3-581BB3   None  None
299502  91e414d5c77a1c0d7985ce50b2ee4758  a9d56ca9194b3bd9  046577a84577e136    2          /api/v1/inventory/{productId}  ...  {"serviceType":"1010","otel.scope.name":"io.op...             F21C6CFACBEB01A3-581BB3   None  None
299503  4e145455195946abe27324e2904e8df5  37ab8844bfdd7c66  d39654846a477527    2          /api/v1/inventory/{productId}  ...  {"serviceType":"1010","otel.scope.name":"io.op...             F21C6CFACBEB01A3-581BB3   None  None
299504  c718073ccece5381b20e17a89da3a7e3  ca6e2889082583af  e8445ad05b97e4aa    2          /api/v1/inventory/{productId}  ...  {"serviceType":"1010","otel.scope.name":"io.op...             F21C6CFACBEB01A3-581BB3   None  None
...                                  ...               ...               ...  ...                                    ...  ...                                                ...        ...                      ...    ...   ...
499509  f30cf0cd95a6190f2efd6c02e332fb8d  6d1cbe8214e0e937  ce22566fe263c278    1  executing api route (pages) /api/cart  ...  {"http.status_code":"200","otel.scope.name":"n...             BAEAC2AE47649F72-58302A   None  None
499510  f30cf0cd95a6190f2efd6c02e332fb8d  ce22566fe263c278  e9b116bb1042d5b9    2                         POST /api/cart  ...  {"otel.scope.name":"next.js","rpc":"POST /api/...             BAEAC2AE47649F72-58302A   None  None
499511  f30cf0cd95a6190f2efd6c02e332fb8d  f7f0a3b79236b913  56f5769004688c69    2          /api/v1/inventory/{productId}  ...  {"serviceType":"1010","otel.scope.name":"io.op...             0B230113C81866B1-592C99   None  None
499512  f30cf0cd95a6190f2efd6c02e332fb8d  15af66a6f2cd3a5d  f7f0a3b79236b913    3             SELECT inventory.inventory  ...  {"serviceType":"2100","otel.scope.name":"io.op...             0B230113C81866B1-592C99   None  None
499513  f30cf0cd95a6190f2efd6c02e332fb8d  e1068a8313228a90  f7f0a3b79236b913    1         HikariDataSource.getConnection  ...  {"serviceType":"6060","otel.scope.name":"io.op...             0B230113C81866B1-592C99   None  None

[32200 rows x 21 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

# Load the logs.parquet file
logs_file_path = f'{task_directory}/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Ensure '_time_' is in a consistent datetime format
logs_data['_time_'] = pd.to_datetime(logs_data['_time_'], errors='coerce')

# Filter logs within the alert time window
filtered_logs = logs_data[
    (logs_data['_time_'] >= alert_start) & (logs_data['_time_'] <= alert_end)
]

# Focus on logs related to the 'inventory' service
inventory_logs = filtered_logs[
    filtered_logs['_container_name_'].str.contains('inventory', case=False, na=False)
]

# Look for error messages or anomalies in the logs
inventory_error_logs = inventory_logs[
    inventory_logs['content'].str.contains('error|exception|fail|critical', case=False, na=False)
]

inventory_error_logs

```
Out[7]:
```
No error messages or anomalies were found in the logs related to the `inventory` service during the alert time window. This suggests that the root cause may not be directly observable in the service logs, and further investigation into other telemetry data or upstream/downstream dependencies may be needed.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, _container_ip_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```